In [1]:
import cv2
import numpy as np
import pickle
import os
from keras.models import load_model

In [2]:
face_classifier = cv2.CascadeClassifier(
    cv2.data.haarcascades + 'haarcascade_frontalface_default.xml'
)

model = load_model('emotion_model.keras')

data_dir = os.path.join(os.getcwd(), 'data')
with open(os.path.join(data_dir, 'label_encoder.p'), 'rb') as f:
    le = pickle.load(f)

emotion_labels = le.classes_
print(f'Emotions: {emotion_labels}')

Emotions: ['angry' 'happy' 'neutral' 'sad' 'surprised']


In [3]:
def preprocess_face(face_img):
    gray = cv2.cvtColor(face_img, cv2.COLOR_BGR2GRAY)
    resized = cv2.resize(gray, (100, 100))
    equalized = cv2.equalizeHist(resized)
    processed = equalized.reshape(1, 100, 100, 1) / 255.0
    return processed

EMOTION_COLORS = {
    'angry': (0, 0, 255),
    'happy': (0, 255, 0),
    'neutral': (255, 255, 0),
    'sad': (255, 0, 0),
    'surprised': (0, 255, 255)
}

In [4]:
cap = cv2.VideoCapture(0)

if  cap.isOpened():
    print('Cannot open webcam. Trying IP camera...')
    import urllib.request
    USE_IP_CAMERA = True
    IP_URL = 'http://10.192.35.82:8080/shot.jpg'
else:
    USE_IP_CAMERA = False
    print('Webcam opened successfully')

while True:
    if USE_IP_CAMERA:
        try:
            img_resp = urllib.request.urlopen(IP_URL)
            img_arr = np.array(bytearray(img_resp.read()), np.uint8)
            frame = cv2.imdecode(img_arr, -1)
        except:
            continue
    else:
        ret, frame = cap.read()
        if not ret:
            break

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    faces = face_classifier.detectMultiScale(gray, scaleFactor=1.3, minNeighbors=5)

    for (x, y, w, h) in faces:
        face_roi = frame[y:y+h, x:x+w]

        processed = preprocess_face(face_roi)
        predictions = model.predict(processed, verbose=0)
        emotion_idx = np.argmax(predictions)
        emotion = emotion_labels[emotion_idx]
        confidence = predictions[0][emotion_idx] * 100

        color = EMOTION_COLORS.get(emotion, (255, 255, 255))

        cv2.rectangle(frame, (x, y), (x+w, y+h), color, 3)

        label = f'{emotion}: {confidence:.1f}%'
        (text_w, text_h), baseline = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.8, 2)
        cv2.rectangle(frame, (x, y - text_h - 10), (x + text_w, y), color, -1)
        cv2.putText(frame, label, (x, y - 5),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)

    cv2.imshow('Emotion Detection', frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

if not USE_IP_CAMERA:
    cap.release()
cv2.destroyAllWindows()
print('Camera closed')

Cannot open webcam. Trying IP camera...
Camera closed
